# LeetCode 3519 — Count Numbers with Non-Decreasing Digits bằng Python

- Link: https://leetcode.com/problems/count-numbers-with-non-decreasing-digits/
- Độ khó: **Hard**
- Chủ đề: **Digit DP** (quy hoạch động theo chữ số), **đổi cơ số** số rất lớn, **tổ hợp chia kẹo** (stars and bars)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countNumbers(self, l: str, r: str, b: int) -> int:
```

## 1. Mục tiêu học

1. Đổi một số thập phân **tới 100 chữ số** sang cơ số `b` (Python có số nguyên lớn).
2. Đưa bài đếm trên đoạn `[l, r]` về `F(r) − F(l − 1)` với `F(N)` = số số trong `[0, N]` thỏa điều kiện.
3. Viết **digit DP** với trạng thái `(vị trí, chữ số trước, còn bám sát N không, đã bắt đầu chưa)`.
4. Rút gọn bằng **tổ hợp**: số dãy không giảm độ dài `L` từ `m` giá trị là `C(L + m − 1, L)`.
5. Kiểm chứng bằng vét cạn với đoạn nhỏ.

## 2. Đề bài & ví dụ

Cho hai số nguyên `l`, `r` dạng **chuỗi thập phân** và cơ số `b`. Đếm số nguyên trong `[l, r]` mà khi viết ở cơ số `b`, các chữ số (từ trái sang phải) **không giảm**. Trả về kết quả modulo `10^9 + 7`.

**Ví dụ 1:** `l = "23", r = "28", b = 8` → `3` (cơ số 8: `27, 30, 31, 32, 33, 34`; hợp lệ: `27, 33, 34`).

**Ví dụ 2:** `l = "2", r = "7", b = 2` → `2` (`10, 11, 100, 101, 110, 111`; hợp lệ `11, 111`).

**Ràng buộc:** `1 ≤ len(l) ≤ len(r) ≤ 100`, `2 ≤ b ≤ 10`, không có số 0 đứng đầu, `l ≤ r`.

In [1]:
MOD = 10**9 + 7
examples = [("23", "28", 8, 3), ("2", "7", 2, 2)]
for l, r, b, exp in examples:
    print(l, r, b, '-> kỳ vọng', exp)

23 28 8 -> kỳ vọng 3
2 7 2 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Đổi sang cơ số `b`

Chia liên tiếp cho `b`, lấy dư, đảo ngược. Python xử lý số 100 chữ số dễ dàng.

In [2]:
def to_base(x, b):
    if x == 0:
        return [0]
    ds = []
    while x:
        x, d = divmod(x, b)
        ds.append(d)
    return ds[::-1]

print([to_base(v, 8) for v in range(23, 29)])
print(len(to_base(10**100 - 1, 2)), 'chữ số nhị phân cho số 100 chữ số thập phân')

[[2, 7], [3, 0], [3, 1], [3, 2], [3, 3], [3, 4]]
333 chữ số nhị phân cho số 100 chữ số thập phân


### 3.2. Kiểm tra không giảm

In [3]:
def non_decreasing(ds):
    return all(ds[i] <= ds[i + 1] for i in range(len(ds) - 1))

print([v for v in range(23, 29) if non_decreasing(to_base(v, 8))])

[23, 27, 28]


### 3.3. Đếm trên đoạn bằng hiệu hai tiền tố

`count([l, r]) = F(r) − F(l − 1)`, với `F(N)` = số `x ∈ [0, N]` thỏa điều kiện (quy ước `F(−1) = 0`). Số `0` (một chữ số `0`) cũng "không giảm" — hiệu `F(r) − F(l − 1)` tự loại nó khi `l ≥ 1`.

### 3.4. Tổ hợp chia kẹo

Số dãy **không giảm** độ dài `L` lấy giá trị trong một tập `m` phần tử có thứ tự = số multiset cỡ `L` = `C(L + m − 1, L)`.

In [4]:
from math import comb
from itertools import combinations_with_replacement
L, m = 3, 4
print(comb(L + m - 1, L), len(list(combinations_with_replacement(range(m), L))))

20 20


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Không thể duyệt `10^{100}` số. Nhưng điều kiện "không giảm" chỉ phụ thuộc **chữ số trước** — đây đúng là kiểu bài **digit DP**: duyệt các chữ số của `N` từ trái sang phải, tại mỗi vị trí chọn chữ số `d`:

- phải `d ≥ chữ số trước` (nếu đã bắt đầu);
- nếu còn **bám sát** `N` (`tight`), `d ≤` chữ số tương ứng của `N`;
- số 0 đứng đầu (chưa bắt đầu) không tính vào điều kiện.

Tính tay `F(28)` với `b = 8`: `28 = 34₈`. Hai chữ số `(a, c)` với `a ≥ 1`, `a ≤ c`, và `≤ 34₈`: `a = 1` ⇒ `c ∈ 1..7` (7 số), `a = 2` ⇒ `c ∈ 2..7` (6), `a = 3` ⇒ `c ∈ 3..4` (2). Một chữ số: `0..7` (8 số). Tổng `23`. Tương tự `F(22) = 20` ⇒ đáp án `3` ✓.

In [5]:
def F_brute(N, b):
    return sum(1 for x in range(N + 1) if non_decreasing(to_base(x, b))) if N >= 0 else 0

print('F(28) =', F_brute(28, 8), '| F(22) =', F_brute(22, 8), '| hiệu =', F_brute(28, 8) - F_brute(22, 8))

F(28) = 23 | F(22) = 20 | hiệu = 3


## 5. Cách 1 — Brute Force: duyệt mọi số trong `[l, r]`

In [6]:
def count_brute(l, r, b):
    return sum(1 for x in range(int(l), int(r) + 1) if non_decreasing(to_base(x, b))) % MOD

for l, r, b, exp in examples:
    print(count_brute(l, r, b), '| kỳ vọng', exp)

3 | kỳ vọng 3
2 | kỳ vọng 2


**Độ phức tạp:** `O((r − l) · log_b r)` — vô vọng khi `r` có 100 chữ số.

**Nút thắt:** số lượng số quá lớn; nhưng "trạng thái" cần nhớ khi đọc từng chữ số rất nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

**Digit DP** cho `F(N)`: gọi `D = to_base(N, b)` (độ dài `m`). Hàm `go(i, prev, tight, started)`:

- `i = m` ⇒ trả 1 (đã tạo một số hợp lệ; số toàn chữ 0 là số 0).
- Chữ số tối đa `hi = D[i]` nếu `tight` ngược lại `b − 1`.
- Duyệt `d` từ `0` tới `hi`:
  - chưa bắt đầu và `d = 0` ⇒ vẫn chưa bắt đầu;
  - ngược lại cần `d ≥ prev` (nếu đã bắt đầu); chuyển sang `(i+1, d, tight ∧ d = hi, True)`.

Số trạng thái `m · b · 2 · 2` ≈ `333 · 10 · 4` — rất nhỏ.

**Tổ hợp (cách 3):** khi không còn bám sát, phần còn lại độ dài `L` chỉ cần là dãy không giảm với giá trị `≥ prev` ⇒ `C(L + (b − prev) − 1, L)` — không cần DP sâu hơn.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Digit DP có nhớ (lời giải chính)

In [7]:
from functools import lru_cache

def F_dp(N, b):
    if N < 0:
        return 0
    D = to_base(N, b)
    m = len(D)

    @lru_cache(maxsize=None)
    def go(i, prev, tight, started):
        if i == m:
            return 1
        hi = D[i] if tight else b - 1
        total = 0
        for d in range(hi + 1):
            nt = tight and d == hi
            if not started and d == 0:
                total += go(i + 1, 0, nt, False)
            elif d >= prev:
                total += go(i + 1, d, nt, True)
        return total % MOD

    return go(0, 0, True, False)

def count_dp(l, r, b):
    return (F_dp(int(r), b) - F_dp(int(l) - 1, b)) % MOD

for l, r, b, exp in examples:
    print(count_dp(l, r, b), '| kỳ vọng', exp)

3 | kỳ vọng 3
2 | kỳ vọng 2


Lưu ý: khi chưa bắt đầu ta đặt `prev = 0`, nên chữ số khác 0 đầu tiên luôn `≥ prev` — đúng ý nghĩa "chưa có ràng buộc".

### 7.2. Cách 3 — Tổ hợp theo tiền tố bám sát (không đệ quy)

Đếm `F(N)` = (số hợp lệ có **ít chữ số hơn** `m`) + (số hợp lệ đúng `m` chữ số và `≤ N`) + 1 (số 0).

- Số có đúng `L` chữ số (chữ số đầu `≥ 1`): dãy không giảm từ `{1..b−1}` ⇒ `C(L + b − 2, L)`.
- Số đúng `m` chữ số `≤ N`: đi dọc `D`; tại vị trí `i` (đang bám sát, chữ số trước `p`), chọn `d` từ `max(p, 1 nếu i = 0)` tới `D[i] − 1` rồi phần còn lại tự do: `C(rest + (b − d) − 1, rest)`. Nếu `D[i] < p` thì dừng (không thể bám sát tiếp). Cuối cùng nếu bản thân `N` không giảm thì cộng 1.

In [8]:
def F_comb(N, b):
    if N < 0:
        return 0
    D = to_base(N, b)
    m = len(D)
    total = 1                                     # số 0
    for L in range(1, m):
        total += comb(L + b - 2, L)
    prev = 1                                      # chữ số đầu phải >= 1
    for i, di in enumerate(D):
        rest = m - i - 1
        for d in range(prev, di):
            total += comb(rest + (b - d) - 1, rest)
        if di < prev:
            break
        prev = di
    else:
        total += 1                                # chính N không giảm
    return total % MOD

def count_comb(l, r, b):
    return (F_comb(int(r), b) - F_comb(int(l) - 1, b)) % MOD

for l, r, b, exp in examples:
    print(count_comb(l, r, b), '| kỳ vọng', exp)

3 | kỳ vọng 3
2 | kỳ vọng 2


**Độ phức tạp:** `O(m · b)` phép tổ hợp (Python `math.comb` trên số vài trăm chữ số — vẫn rất nhanh); có thể thay bằng bảng giai thừa modulo.

### 7.3. Cách 4 — DP lặp theo chữ số trước (bottom-up, không lru_cache)

Giữ hai mảng: `free[p]` = số cách khi **đã thoát** khỏi bám sát và chữ số trước là `p`, và trạng thái bám sát duy nhất. Đi từ trái sang phải, cộng dồn.

In [9]:
def F_iter(N, b):
    if N < 0:
        return 0
    D = to_base(N, b)
    m = len(D)
    free = [0] * b             # free[p]: số tiền tố đã bắt đầu, < N về tiền tố, chữ số cuối p
    not_started = 0            # số tiền tố toàn 0 nhưng đã < N (chỉ xảy ra sau vị trí đầu)
    tight_prev, tight_ok = 0, True
    for i, di in enumerate(D):
        nfree = [0] * b
        # từ free: chọn d >= p tùy ý
        acc = 0
        for d in range(b):
            acc = (acc + free[d]) % MOD
            nfree[d] = acc                        # tổng free[p] với p <= d
        # từ "chưa bắt đầu" (đã nhỏ hơn N): chọn d >= 1 để bắt đầu, hoặc 0 để tiếp tục chưa bắt đầu
        for d in range(1, b):
            nfree[d] = (nfree[d] + not_started) % MOD
        nnot = not_started
        # từ trạng thái bám sát
        if tight_ok:
            started = i > 0                   # D[0] >= 1 nên tiền tố bám sát đã bắt đầu từ vị trí 1
            lo = tight_prev if started else 0
            for d in range(lo, di):
                if not started and d == 0:
                    nnot += 1
                else:
                    nfree[d] = (nfree[d] + 1) % MOD
            if started and di < tight_prev:
                tight_ok = False
            else:
                tight_prev = di
        free, not_started = nfree, nnot % MOD
    return (sum(free) + not_started + (1 if tight_ok else 0)) % MOD

def count_iter(l, r, b):
    return (F_iter(int(r), b) - F_iter(int(l) - 1, b)) % MOD

for l, r, b, exp in examples:
    print(count_iter(l, r, b), '| kỳ vọng', exp)

3 | kỳ vọng 3
2 | kỳ vọng 2


Cách này thể hiện đúng tinh thần digit DP "từng lớp" và chạy `O(m · b)`.

## 8. TRACE từng bước

Trace cách 3 cho `N = 28`, `b = 8` (`D = [3, 4]`): in các nhóm được đếm.

In [10]:
def trace_comb(N, b):
    D = to_base(N, b)
    m = len(D)
    print(f"N = {N} = {D} (cơ số {b})")
    total = 1
    print('  số 0: 1')
    for L in range(1, m):
        c = comb(L + b - 2, L)
        total += c
        print(f"  số có {L} chữ số (không giảm, chữ số đầu >= 1): C({L}+{b}-2, {L}) = {c}")
    prev = 1
    for i, di in enumerate(D):
        rest = m - i - 1
        for d in range(prev, di):
            c = comb(rest + (b - d) - 1, rest)
            total += c
            print(f"  bám sát {D[:i]} rồi chọn {d} < {di} ở vị trí {i}, {rest} chữ số tự do >= {d}: {c}")
        if di < prev:
            print(f"  D[{i}] = {di} < {prev}: không bám sát tiếp được"); break
        prev = di
    else:
        total += 1
        print('  chính N không giảm: +1')
    print('  F =', total)
    return total

trace_comb(28, 8)
print()
trace_comb(22, 8)

N = 28 = [3, 4] (cơ số 8)
  số 0: 1
  số có 1 chữ số (không giảm, chữ số đầu >= 1): C(1+8-2, 1) = 7
  bám sát [] rồi chọn 1 < 3 ở vị trí 0, 1 chữ số tự do >= 1: 7
  bám sát [] rồi chọn 2 < 3 ở vị trí 0, 1 chữ số tự do >= 2: 6
  bám sát [3] rồi chọn 3 < 4 ở vị trí 1, 0 chữ số tự do >= 3: 1
  chính N không giảm: +1
  F = 23

N = 22 = [2, 6] (cơ số 8)
  số 0: 1
  số có 1 chữ số (không giảm, chữ số đầu >= 1): C(1+8-2, 1) = 7
  bám sát [] rồi chọn 1 < 2 ở vị trí 0, 1 chữ số tự do >= 1: 7
  bám sát [2] rồi chọn 2 < 6 ở vị trí 1, 0 chữ số tự do >= 2: 1
  bám sát [2] rồi chọn 3 < 6 ở vị trí 1, 0 chữ số tự do >= 3: 1
  bám sát [2] rồi chọn 4 < 6 ở vị trí 1, 0 chữ số tự do >= 4: 1
  bám sát [2] rồi chọn 5 < 6 ở vị trí 1, 0 chữ số tự do >= 5: 1
  chính N không giảm: +1
  F = 20


20

`F(28) − F(22) = 23 − 20 = 3` ✓.

## 9. Code nộp LeetCode

In [11]:
class Solution:
    def countNumbers(self, l: str, r: str, b: int) -> int:
        MOD = 10**9 + 7

        def F(N):
            if N < 0:
                return 0
            D = []
            x = N
            while x:
                x, d = divmod(x, b)
                D.append(d)
            D = D[::-1] or [0]
            m = len(D)

            @lru_cache(maxsize=None)
            def go(i, prev, tight, started):
                if i == m:
                    return 1
                hi = D[i] if tight else b - 1
                total = 0
                for d in range(hi + 1):
                    nt = tight and d == hi
                    if not started and d == 0:
                        total += go(i + 1, 0, nt, False)
                    elif d >= prev:
                        total += go(i + 1, d, nt, True)
                return total % MOD

            res = go(0, 0, True, False)
            go.cache_clear()
            return res

        return (F(int(r)) - F(int(l) - 1)) % MOD

for l, r, b, exp in examples:
    got = Solution().countNumbers(l, r, b)
    print(got, 'OK' if got == exp else 'SAI')

3 OK
2 OK


## 10. Vì sao thuật toán đúng

1. **Hiệu tiền tố:** `{x : l ≤ x ≤ r} = [0, r] \ [0, l−1]`.
2. **Song ánh số ↔ dãy chữ số:** mỗi `x ∈ [0, N]` tương ứng duy nhất một dãy `m` chữ số (thêm 0 đứng đầu) `≤ D` theo thứ tự từ điển. Digit DP liệt kê đúng các dãy này nhờ ràng buộc `tight`.
3. **Điều kiện không giảm** chỉ áp cho các chữ số **sau** chữ số khác 0 đầu tiên (0 đứng đầu không phải chữ số thật) — trạng thái `started` tách bạch điều đó.
4. **Ghi nhớ đúng:** số cách hoàn thành chỉ phụ thuộc `(i, prev, tight, started)`.
5. **Tổ hợp (cách 3):** khi đã nhỏ hơn `N` ở vị trí `i`, các chữ số còn lại tự do miễn không giảm và `≥ d` ⇒ `C(rest + (b − d) − 1, rest)` (chia kẹo trên `b − d` giá trị).

## 11. Phân tích Time / Space Complexity

Gọi `m = log_b(r) ≤ 333` (khi `b = 2`).

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt | `O((r−l) m)` | `O(m)` |
| 2. Digit DP có nhớ | `O(m · b · 4 · b)` | `O(m · b · 4)` |
| 3. Tổ hợp | `O(m · b)` lần `comb` | `O(m)` |
| 4. DP lặp | `O(m · b)` | `O(b)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [12]:
impls = {'dp': count_dp, 'comb': count_comb, 'iter': count_iter,
         'submit': lambda l, r, b: Solution().countNumbers(l, r, b)}
edge = [("1", "1", 2, 1), ("1", "9", 10, 9), ("10", "11", 10, 1), ("1", "1000", 10, None), ("5", "5", 3, None)]
for l, r, b, exp in examples + edge:
    br = count_brute(l, r, b)
    if exp is not None:
        assert br == exp
    for name, f in impls.items():
        assert f(l, r, b) == br, (name, l, r, b)
print('Ví dụ đề và ca biên: tất cả đúng. (1..1000, b=10):', count_dp("1", "1000", 10))

Ví dụ đề và ca biên: tất cả đúng. (1..1000, b=10): 219


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [13]:
import random
random.seed(3519)
cnt = 0
for _ in range(400):
    b = random.randint(2, 10)
    lo = random.randint(1, 3000)
    hi = lo + random.randint(0, 3000)
    exp = count_brute(str(lo), str(hi), b)
    for name, f in impls.items():
        assert f(str(lo), str(hi), b) == exp, (name, lo, hi, b)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'đoạn ngẫu nhiên.')

Khớp vét cạn trên 400 đoạn ngẫu nhiên.


### 12.3. Ca lớn (`r` có 100 chữ số)

In [14]:
import time
random.seed(100)
for b in (2, 3, 7, 10):
    r = str(random.randint(10**99, 10**100 - 1))
    l = str(random.randint(1, int(r)))
    t = time.time(); a = Solution().countNumbers(l, r, b); t1 = time.time() - t
    assert a == count_comb(l, r, b) == count_iter(l, r, b)
    print(f"b={b:>2}: {a} | {t1 * 1000:.1f} ms")

b= 2: 0 | 1.0 ms
b= 3: 210 | 0.9 ms
b= 7: 233356976 | 1.5 ms
b=10: 5050 | 2.6 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Bỏ trạng thái `started` (coi 0 đứng đầu như chữ số thật)

Với điều kiện **không giảm**, 0 đứng đầu vô hại (0 nhỏ nhất, không chặn chữ số sau) nên bỏ `started` vẫn đúng. Nhưng thói quen này gây lỗi ngay khi đổi sang điều kiện **không tăng** (biến thể 2, bài tập 5): 0 đứng đầu sẽ chặn mọi chữ số khác 0 phía sau.

In [15]:
def nonincreasing(ds):
    return all(ds[i] >= ds[i + 1] for i in range(len(ds) - 1))

def F_nonincr_no_started(N, b):
    D = to_base(N, b)
    m = len(D)
    @lru_cache(maxsize=None)
    def go(i, prev, tight):
        if i == m:
            return 1
        hi = D[i] if tight else b - 1
        return sum(go(i + 1, d, tight and d == hi) for d in range(0, min(prev, hi) + 1))   # SAI: 0 đầu chặn
    return go(0, b - 1, True)

def F_nonincr_brute(N, b):
    return sum(1 for x in range(N + 1) if nonincreasing(to_base(x, b)))

def F_nondecr_no_started(N, b):
    D = to_base(N, b)
    m = len(D)
    @lru_cache(maxsize=None)
    def go(i, prev, tight):
        if i == m:
            return 1
        hi = D[i] if tight else b - 1
        return sum(go(i + 1, d, tight and d == hi) for d in range(prev, hi + 1))
    return go(0, 0, True)

print('không giảm, bỏ started :', F_nondecr_no_started(1000, 10), '| vét cạn:', F_brute(1000, 10), '(vô hại)')
print('không tăng, bỏ started :', F_nonincr_no_started(100, 10), '| vét cạn:', F_nonincr_brute(100, 10), '(SAI)')

không giảm, bỏ started : 220 | vét cạn: 220 (vô hại)
không tăng, bỏ started : 2 | vét cạn: 65 (SAI)


**Vì sao sai:** số `50` được xem như `050` — dãy `0, 5, 0` không "không tăng", nên bị loại oan. Trạng thái `started` cho biết các chữ số 0 phía trước **không phải** chữ số thật.

### Lỗi 2 — Đổi cơ số trên **chuỗi thập phân** từng ký tự (sai bản chất)

In [16]:
def wrong_digits(s, b):
    return [int(c) % b for c in s]         # SAI: chỉ lấy từng chữ số thập phân mod b

print('sai :', wrong_digits("28", 8), '| đúng:', to_base(28, 8))

sai : [2, 0] | đúng: [3, 4]


**Vì sao sai:** phải đổi **giá trị** của số sang cơ số `b`, không phải từng chữ số.

### Lỗi 3 — Tính `F(r) − F(l)` (quên trừ 1)

In [17]:
print('sai :', (F_dp(28, 8) - F_dp(23, 8)) % MOD, '| đúng:', count_dp("23", "28", 8))

sai : 2 | đúng: 3


**Vì sao sai:** `F(l)` chứa cả `l`; muốn giữ `l` phải trừ `F(l − 1)`.

### Lỗi 4 — Quên modulo khi trừ (kết quả âm)

In [18]:
a, c = 5, 7
print('trừ thô:', a - c, '| có % MOD:', (a - c) % MOD)

trừ thô: -2 | có % MOD: 1000000005


**Vì sao sai:** sau khi lấy modulo, `F(r) mod p` có thể nhỏ hơn `F(l−1) mod p`; phải `% MOD` lại sau phép trừ.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm số trong `[l, r]` (rất lớn) thỏa tính chất theo chữ số" ⇒ **digit DP** + `F(r) − F(l−1)`.
- Trạng thái thường gặp: `(vị trí, thông tin chữ số trước/tổng/dư, tight, started)`.
- Tính chất "dãy chữ số không giảm" ⇒ đếm bằng **chia kẹo** khi phần còn lại tự do.
- Chuỗi số lớn ⇒ Python `int(s)` rồi đổi cơ số.

## 15. Biến thể của bài

1. Chữ số **tăng ngặt** ⇒ chia kẹo thành `C(b − d − 1, rest)` (chọn tập con).
2. Chữ số **không tăng** ⇒ cần `started` (0 đứng đầu); hoặc đếm phần bù.
3. Thêm điều kiện **tổng chữ số chia hết cho k** ⇒ thêm chiều số dư.
4. Bài liên quan: LC 902 *Numbers At Most N Given Digit Set*, LC 1012 *Numbers With Repeated Digits*, LC 2376.

In [19]:
# Biến thể 1: chữ số tăng ngặt trong cơ số b, đếm trên [0, N]
def F_strict(N, b):
    D = to_base(N, b)
    m = len(D)
    @lru_cache(maxsize=None)
    def go(i, prev, tight, started):
        if i == m:
            return 1
        hi = D[i] if tight else b - 1
        t = 0
        for d in range(hi + 1):
            nt = tight and d == hi
            if not started and d == 0:
                t += go(i + 1, 0, nt, False)
            elif not started or d > prev:
                t += go(i + 1, d, nt, True)
        return t
    return go(0, 0, True, False)

def strictly_inc(ds):
    return all(ds[i] < ds[i + 1] for i in range(len(ds) - 1))

for N, b in [(500, 10), (200, 4), (63, 2)]:
    assert F_strict(N, b) == sum(1 for x in range(N + 1) if strictly_inc(to_base(x, b)))
print('Biến thể tăng ngặt: khớp vét cạn; F(500) cơ số 10 =', F_strict(500, 10))

Biến thể tăng ngặt: khớp vét cạn; F(500) cơ số 10 = 120


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1 | Duyệt | `O((r−l) m)` | Oracle |
| 2 | Digit DP có nhớ | `O(m b^2)` | **Nộp bài** (khuôn chuẩn) |
| 3 | Tổ hợp chia kẹo | `O(m b)` | Ngắn, nhanh |
| 4 | DP lặp | `O(m b)` | Không đệ quy |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `countNumbers("1", "20", 10)`.

**Bài 2 (tính tay):** `countNumbers("1", "15", 2)`.

**Bài 3 (điền TODO):** viết `count_length(L, b)` — số số có đúng `L` chữ số (cơ số `b`) có chữ số không giảm.

**Bài 4 (tự viết lại):** biến thể 3 — thêm điều kiện tổng chữ số chia hết cho `k`.

**Bài 5 (mini challenge):** biến thể 2 — chữ số **không tăng**.

In [20]:
def count_length_todo(L, b):
    # TODO: C(L + b - 2, L)
    raise NotImplementedError

def count_sum_div_todo(l, r, b, k):
    # TODO
    raise NotImplementedError

def count_nonincreasing_todo(l, r, b):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `1..9` đều hợp lệ (9 số). (2) `10..20`: `11, 12, …, 19` (9 số). (3) `18`.

**Bài 2.** (1) Nhị phân không giảm: `1, 11, 111, 1111`. (2) `≤ 15 = 1111₂`. (3) `4`.

**Bài 3.** (1) Chữ số trong `{1..b−1}`. (2) Chia kẹo `m = b − 1`. (3) So với vét cạn.

**Bài 4.** (1) Thêm `s = tổng mod k` vào trạng thái. (2) Kết thúc: `started` và `s = 0` (hoặc số 0). (3) Vét cạn.

**Bài 5.** (1) `prev` khởi tạo "vô cực" khi chưa bắt đầu. (2) Chọn `d ≤ prev`. (3) Cần `started`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2, 3
a1 = Solution().countNumbers("1", "20", 10)
a2 = Solution().countNumbers("1", "15", 2)
assert a1 == 18 == count_brute("1", "20", 10) and a2 == 4 == count_brute("1", "15", 2)

def count_length(L, b):
    return comb(L + b - 2, L)

for b in range(2, 11):
    for L in range(1, 5):
        lo, hi = b ** (L - 1), b ** L - 1
        assert count_length(L, b) == sum(1 for x in range(lo, hi + 1) if non_decreasing(to_base(x, b)))
print('Bài 1:', a1, '| Bài 2:', a2, '| Bài 3: đúng')

Bài 1: 18 | Bài 2: 4 | Bài 3: đúng


In [22]:
# Bài 4: thêm tổng chữ số chia hết cho k
def F_sum(N, b, k):
    if N < 0:
        return 0
    D = to_base(N, b)
    m = len(D)
    @lru_cache(maxsize=None)
    def go(i, prev, tight, started, s):
        if i == m:
            return 1 if (not started or s == 0) else 0
        hi = D[i] if tight else b - 1
        t = 0
        for d in range(hi + 1):
            nt = tight and d == hi
            if not started and d == 0:
                t += go(i + 1, 0, nt, False, 0)
            elif d >= prev:
                t += go(i + 1, d, nt, True, (s + d) % k)
        return t
    return go(0, 0, True, False, 0)

def count_sum_div(l, r, b, k):
    return F_sum(int(r), b, k) - F_sum(int(l) - 1, b, k)

for l, r, b, k in [(1, 500, 10, 3), (1, 300, 4, 2), (50, 900, 7, 5)]:
    br = sum(1 for x in range(l, r + 1) if non_decreasing(to_base(x, b)) and sum(to_base(x, b)) % k == 0)
    assert count_sum_div(str(l), str(r), b, k) == br
print('Bài 4: khớp vét cạn')

Bài 4: khớp vét cạn


In [23]:
# Bài 5: không tăng
def F_nonincr(N, b):
    if N < 0:
        return 0
    D = to_base(N, b)
    m = len(D)
    @lru_cache(maxsize=None)
    def go(i, prev, tight, started):
        if i == m:
            return 1
        hi = D[i] if tight else b - 1
        t = 0
        for d in range(hi + 1):
            nt = tight and d == hi
            if not started and d == 0:
                t += go(i + 1, 0, nt, False)
            elif not started or d <= prev:
                t += go(i + 1, d, nt, True)
        return t
    return go(0, 0, True, False)

def count_nonincreasing(l, r, b):
    return F_nonincr(int(r), b) - F_nonincr(int(l) - 1, b)

for l, r, b in [(1, 1000, 10), (1, 500, 3), (100, 2000, 6)]:
    assert count_nonincreasing(str(l), str(r), b) == sum(1 for x in range(l, r + 1) if nonincreasing(to_base(x, b)))
print('Bài 5: khớp vét cạn; không tăng trong [1, 1000] cơ số 10:', count_nonincreasing("1", "1000", 10))

Bài 5: khớp vét cạn; không tăng trong [1, 1000] cơ số 10: 283


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
F(N): D = chữ số của N ở cơ số b
go(i, prev, tight, started):
    i == m -> 1
    for d in 0..(D[i] if tight else b-1):
        chưa bắt đầu và d == 0 -> go(i+1, 0, tight & d==hi, False)
        d >= prev             -> go(i+1, d, tight & d==hi, True)
đáp án = (F(r) - F(l-1)) % MOD
```

**Checklist:**

- [ ] Đổi **giá trị** sang cơ số `b` (không đổi từng chữ số)?
- [ ] `F(r) − F(l − 1)` và `% MOD` sau phép trừ?
- [ ] `tight` cập nhật `tight and d == hi`?
- [ ] Hiểu khi nào cần `started` (không tăng: cần; không giảm: vô hại)?
- [ ] Đối chiếu vét cạn trên đoạn nhỏ?